# 33 · E4 Imprint Pre-bias Test · 真器件

> ⚠️ 真连 B1500，接 pFeFET 器件。前置：E1 (30) PASS。

**目的 (H4 Imprint)**：检验预偏置能否系统性移动 MW 中心——
若 +2V 预偏置使 MW 向正方向偏移，说明铁电矫顽电场随历史偏移（imprint）。

序列：pre_bias(V_pre, t_pre) → neutral_hold(1ms) → Write(±5V,100μs) → delay → 3pt-Read

扫参：V_pre ∈ {+2, −2, 0}V × t_pre ∈ {1ms, 10ms, 100ms, 1s} × delay ∈ {10μs, 10s}

测试人：**yhzang**

In [ ]:
import sys, os, time, random, datetime, itertools
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path, autodetect_visa_addr,
    autodetect_wgfmu_chan, RealWgfmuBackend,
)

print("Python:", sys.version.split()[0])
print("project root:", ROOT)

In [ ]:
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

backend = RealWgfmuBackend()
backend.load()
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)
channel_ids = backend.get_channel_ids()
print(f"detected channels: {channel_ids}")

GATE_CH  = autodetect_wgfmu_chan(backend, prefer=201)
DRAIN_CH = 202
assert DRAIN_CH in channel_ids
print(f"✅ gate={GATE_CH}, drain={DRAIN_CH}")
backend.close_session()

In [ ]:
# ── USER-EDITABLE PARAMETERS ──────────────────────────────────
QUICK_MODE  = True       # True: V_pre=[+2,-2], t_pre=[1ms,100ms], delay=[10us], 2 reps

DEVICE_ID   = "dev001"
GEOMETRY    = "W5L10"

V_ERS       = +5.0
V_PGM       = -5.0
T_WRITE     = 100e-6
T_RISE_FALL = 100e-9
T_NEUTRAL   = 1e-3       # neutral hold after pre-bias, before write

V_PRE_QUICK  = [+2.0, -2.0, 0.0]
V_PRE_FULL   = [+2.0, -2.0, 0.0]

T_PRE_QUICK  = [1e-3, 100e-3]                    # 1ms, 100ms
T_PRE_FULL   = [1e-3, 10e-3, 100e-3, 1.0]        # 1ms, 10ms, 100ms, 1s

DELAYS_QUICK = [10e-6]                            # 10us only
DELAYS_FULL  = [10e-6, 10.0]                      # 10us and 10s

VG_READS    = [-0.2, 0.0, +0.2]
VD_READ     = 0.05
T_READ      = 5e-6
N_PTS_READ  = 5

N_REPS_QUICK = 2
N_REPS_FULL  = 5
RANDOM_SEED  = 42
# ── END PARAMETERS ────────────────────────────────────────────

v_pres  = V_PRE_QUICK  if QUICK_MODE else V_PRE_FULL
t_pres  = T_PRE_QUICK  if QUICK_MODE else T_PRE_FULL
delays  = DELAYS_QUICK if QUICK_MODE else DELAYS_FULL
N_REPS  = N_REPS_QUICK if QUICK_MODE else N_REPS_FULL
TAG     = "quick" if QUICK_MODE else "full"

combos = list(itertools.product(v_pres, t_pres, ["ERS", "PGM"], delays))
total_shots = len(combos) * N_REPS
# Rough time estimate
t_est = sum(t_pre + T_NEUTRAL + T_WRITE + d + 3*(T_READ+2*T_RISE_FALL)
            for _, t_pre, _, d in combos) * N_REPS
print(f"Mode  : {TAG}")
print(f"Combos: {len(combos)} = {len(v_pres)}×{len(t_pres)}×2 states×{len(delays)} delays")
print(f"Total shots: {total_shots}")
print(f"Est. time  : {t_est:.1f} s ({t_est/60:.1f} min)")

In [ ]:
def _run_imprint_shot(
    backend, gate_ch, drain_ch,
    v_pre, t_pre, t_neutral,
    v_write, t_write, t_rise_fall,
    t_delay, vg_reads, vd_read, t_read, n_pts=5,
) -> list:
    """Pre-bias → neutral → write → delay → read.

    Gate starts at v_pre; drain holds 0V throughout pre+neutral+write,
    then switches to vd_read for the read pulses.
    """
    T_GAP  = 100e-9
    GUARD  = 200e-9
    t_delay = max(t_delay, 200e-9)
    t_pre   = max(t_pre,   200e-9)

    backend.clear()

    # Gate pattern: starts at v_pre
    backend.create_pattern("gp", float(v_pre))
    backend.add_vector("gp", t_pre,       float(v_pre))  # pre-bias hold
    backend.add_vector("gp", t_rise_fall, 0.0)           # ramp to 0V
    backend.add_vector("gp", t_neutral,   0.0)           # neutral hold
    backend.add_vector("gp", t_rise_fall, float(v_write))# rise to write
    backend.add_vector("gp", t_write,     float(v_write))# write hold
    backend.add_vector("gp", t_rise_fall, 0.0)           # fall
    backend.add_vector("gp", t_delay,     0.0)           # delay

    # Pre-read total time (from pattern start)
    t_pre_read = (t_pre + t_rise_fall + t_neutral +
                  t_rise_fall + t_write + t_rise_fall + t_delay)

    # Drain pattern: 0V until reads, then vd_read
    backend.create_pattern("dp", 0.0)
    backend.add_vector("dp", t_pre_read, 0.0)

    meas_win = t_read - GUARD
    interval = meas_win / max(n_pts, 1)
    average  = interval * 0.8

    t_cursor = t_pre_read
    read_t0s = []
    for i, vg in enumerate(vg_reads):
        t_cursor += t_rise_fall
        read_t0s.append(t_cursor)
        t_cursor += t_read + t_rise_fall
        backend.add_vector("gp", t_rise_fall, float(vg))
        backend.add_vector("gp", t_read,       float(vg))
        backend.add_vector("gp", t_rise_fall,  0.0)
        backend.add_vector("dp", t_rise_fall, vd_read)
        backend.add_vector("dp", t_read,       vd_read)
        backend.add_vector("dp", t_rise_fall,  0.0)
        if i < len(vg_reads) - 1:
            t_cursor += T_GAP
            backend.add_vector("gp", T_GAP, 0.0)
            backend.add_vector("dp", T_GAP, 0.0)

    for i in range(len(vg_reads)):
        t_ev = read_t0s[i] + GUARD
        backend.set_measure_event("gp", f"g{i}", t_ev, n_pts, interval, average, "averaged")
        backend.set_measure_event("dp", f"d{i}", t_ev, n_pts, interval, average, "averaged")

    backend.add_sequence(gate_ch,  "gp", 1)
    backend.add_sequence(drain_ch, "dp", 1)

    backend.initialize()
    for ch in [gate_ch, drain_ch]:
        backend.set_operation_mode(ch, "FASTIV")
        backend.set_force_voltage_range(ch, "AUTO")
        backend.set_measure_enabled(ch, True)
        backend.set_measure_mode(ch, "CURRENT")
        backend.set_measure_current_range(ch, "1MA")
    backend.connect(gate_ch)
    backend.connect(drain_ch)

    backend.execute()
    backend.wait_until_completed()

    g_df = backend.get_measure_values(gate_ch)
    d_df = backend.get_measure_values(drain_ch)
    backend.disconnect(gate_ch)
    backend.disconnect(drain_ch)

    g_t = g_df["time_s"].values;  g_v = g_df["value"].values
    d_t = d_df["time_s"].values;  d_v = d_df["value"].values

    results = []
    for i, vg in enumerate(vg_reads):
        t0 = read_t0s[i] + GUARD
        t1 = t0 + meas_win
        d_sub = d_v[(d_t >= t0) & (d_t <= t1)]
        g_sub = g_v[(g_t >= t0) & (g_t <= t1)]
        results.append(dict(
            Vg_read_V=float(vg), Vd_read_V=vd_read,
            Id_mean_A = float(np.nanmean(d_sub)) if len(d_sub) > 0 else float("nan"),
            Id_std_A  = float(np.nanstd(d_sub))  if len(d_sub) > 1 else float("nan"),
            Ig_mean_A = float(np.nanmean(g_sub)) if len(g_sub) > 0 else float("nan"),
            n_d=int(len(d_sub)), n_g=int(len(g_sub)),
        ))
    return results


print("✅ _run_imprint_shot() defined")

In [ ]:
ts_start = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir  = ROOT / "runs" / f"{ts_start}_E4_imprint_{TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
out_csv  = run_dir / "imprint_results.csv"
print(f"Output : {run_dir}")

random.seed(RANDOM_SEED)
combos_run = list(combos)
random.shuffle(combos_run)
print(f"Shuffled {len(combos_run)} combos")

In [ ]:
all_rows = []
seq_id   = 0

backend.open_session(VISA_ADDR)
t_exp0 = time.time()
try:
    print(f"Starting E4 Imprint: {total_shots} shots")

    for rep in range(N_REPS):
        for v_pre, t_pre, state, t_delay in combos_run:
            v_write = V_ERS if state == "ERS" else V_PGM
            shot_timeout = max(30.0, t_pre * 3 + t_delay * 3 + 10.0)
            backend.set_timeout(shot_timeout)

            ts_iso = datetime.datetime.now().isoformat(timespec="seconds")
            t_sh   = time.time()
            note   = ""
            try:
                rr = _run_imprint_shot(
                    backend, GATE_CH, DRAIN_CH,
                    v_pre=v_pre, t_pre=t_pre, t_neutral=T_NEUTRAL,
                    v_write=v_write, t_write=T_WRITE, t_rise_fall=T_RISE_FALL,
                    t_delay=t_delay, vg_reads=VG_READS,
                    vd_read=VD_READ, t_read=T_READ, n_pts=N_PTS_READ,
                )
            except Exception as exc:
                note = f"ERR:{exc}"
                print(f"  !! Vpre={v_pre}V Tpre={t_pre:.1e} {state} d={t_delay:.1e}: {exc}")
                rr = [dict(Vg_read_V=vg, Vd_read_V=VD_READ,
                           Id_mean_A=float("nan"), Id_std_A=float("nan"),
                           Ig_mean_A=float("nan"), n_d=0, n_g=0)
                      for vg in VG_READS]
                for ch in [GATE_CH, DRAIN_CH]:
                    try: backend.disconnect(ch)
                    except Exception: pass
                try: backend.close_session()
                except Exception: pass
                time.sleep(1.0)
                backend.open_session(VISA_ADDR)

            for r in rr:
                all_rows.append(dict(
                    timestamp_iso=ts_iso, device_id=DEVICE_ID,
                    geometry=GEOMETRY, sequence_id=seq_id,
                    repeat_index=rep, state_target=state,
                    v_pre_V=v_pre, t_pre_s=t_pre,
                    delay_s=t_delay,
                    Vg_read_V=r["Vg_read_V"], Vd_read_V=r["Vd_read_V"],
                    Id_mean_A=r["Id_mean_A"], Id_std_A=r["Id_std_A"],
                    Ig_mean_A=r["Ig_mean_A"], note=note,
                ))
            seq_id += 1

            Id0 = rr[1]["Id_mean_A"]
            print(f"  rep={rep} Vpre={v_pre:+.0f}V Tpre={t_pre:.1e} {state:3s} d={t_delay:.1e} | "
                  f"Id(0V)={Id0*1e9:.1f}nA | {time.time()-t_sh:.2f}s")

        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"\n✅ Done in {time.time()-t_exp0:.1f}s. Saved: {out_csv}")

except Exception as exc:
    print(f"\n❌ Failed: {exc}")
    raise
finally:
    for ch in [GATE_CH, DRAIN_CH]:
        try: backend.disconnect(ch)
        except Exception: pass
    backend.close_session()

df = pd.DataFrame(all_rows)
print(df.shape)

In [ ]:
df = pd.read_csv(out_csv)
vg_ref = 0.0
sub    = df[df["Vg_read_V"] == vg_ref]

v_pres_u  = sorted(sub["v_pre_V"].unique())
t_pres_u  = sorted(sub["t_pre_s"].unique())
delays_u  = sorted(sub["delay_s"].unique())

n_drow = len(delays_u)
fig, axes = plt.subplots(n_drow, 1, figsize=(10, 4*n_drow), squeeze=False)

for row, t_del in enumerate(delays_u):
    ax = axes[row, 0]
    sub_d = sub[sub["delay_s"] == t_del]

    ers_g = sub_d[sub_d["state_target"] == "ERS"].groupby(["v_pre_V", "t_pre_s"])["Id_mean_A"].mean()
    pgm_g = sub_d[sub_d["state_target"] == "PGM"].groupby(["v_pre_V", "t_pre_s"])["Id_mean_A"].mean()
    mw_df = (ers_g - pgm_g).rename("MW_A").reset_index()

    for v_pre in v_pres_u:
        m = mw_df[mw_df["v_pre_V"] == v_pre].sort_values("t_pre_s")
        ax.plot(m["t_pre_s"] * 1e3, m["MW_A"] * 1e9,
                "o-", ms=6, label=f"V_pre={v_pre:+.0f}V")
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel("t_pre (ms)")
    ax.set_ylabel("MW (nA)")
    ax.set_title(f"MW vs pre-bias duration | delay={t_del*1e6:.0f}μs")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(f"E4 Imprint – {DEVICE_ID} {GEOMETRY}")
plt.tight_layout()
plt.savefig(run_dir / "imprint_mw_vs_pre.png", dpi=150, bbox_inches="tight")
plt.show()

# Print MW offset at longest t_pre: positive vs negative pre-bias
for t_del in delays_u:
    sub_d = sub[sub["delay_s"] == t_del]
    ers_g = sub_d[sub_d["state_target"] == "ERS"].groupby(["v_pre_V", "t_pre_s"])["Id_mean_A"].mean()
    pgm_g = sub_d[sub_d["state_target"] == "PGM"].groupby(["v_pre_V", "t_pre_s"])["Id_mean_A"].mean()
    mw_df = (ers_g - pgm_g).rename("MW_A").reset_index()
    t_pre_max = mw_df["t_pre_s"].max()
    long = mw_df[mw_df["t_pre_s"] == t_pre_max][["v_pre_V", "MW_A"]].set_index("v_pre_V")
    print(f"delay={t_del:.1e}s, t_pre={t_pre_max:.1e}s: MW by V_pre =\n{long * 1e9}")

## 通过标准

- [ ] 无硬件错误，CSV 完整
- [ ] +2V 预偏置 vs −2V 预偏置的 MW 有系统性差异 → imprint 存在
- [ ] 0V 预偏置的 MW 与 E1 基准值一致

**结论记录**：MW(+2V pre) − MW(−2V pre) = ___ nA at t_pre=1s, delay=10μs